In [ ]:
import os
cache_dir = "/kaggle/input/datasets/awsaf49/ade20k-dataset/ADEChallengeData2016"
print(os.listdir(cache_dir))
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  

In [ ]:
!pip install -q huggingface_hub
!pip install -q transformers

In [ ]:
import sys, os, time, glob, json, warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm


print("Installing core packages...")
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q datasets accelerate evaluate pillow tqdm

print("Installing ONNX tools...")
!pip install -q onnx onnxsim


if not os.path.exists("/kaggle/working/efficientvit"):
    print("Cloning EfficientViT repository...")
    !git clone https://github.com/mit-han-lab/efficientvit.git /kaggle/working/efficientvit
else:
    print("EfficientViT repository already exists.")


sys.path.insert(0, '/kaggle/working/efficientvit')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from datasets import load_dataset
from torchvision import transforms
from PIL import Image

from efficientvit.models.efficientvit import efficientvit_seg_b2

print("Environment ready. EfficientViT modules imported successfully!")

In [ ]:
import time
import os

CONFIG = {
    "num_classes": 150,
    "image_size": 384,
    "batch_size": 4,                 
    "num_epochs": 10,                
    "lr": 6e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "data_split": 1.0,               
    "val_fraction": 0.1,
    "checkpoint_interval": 1,        
    "max_grad_norm": 1.0,
    "use_amp": True,
    "session_start": time.time(),
    "max_session_hours": 11.5,       
    "precache": False,               
    "num_workers": 4,                
    "persistent_workers": True,      
    "prefetch_factor": 2,            
    "pin_memory": True
}


ADE20K_ROOT = "/kaggle/input/datasets/awsaf49/ade20k-dataset/ADEChallengeData2016"
DATASET_SOURCE = "local"
USE_HF_DATASET = False


import datetime
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
cache_tag = "nocache"
split_tag = f"{int(CONFIG['data_split']*100)}pct"
epoch_tag = f"{CONFIG['num_epochs']}ep"
folder_name = f"efficientvit_ade20k_{split_tag}_{epoch_tag}_{cache_tag}_{timestamp}"
CONFIG["output_dir"] = f"/kaggle/working/{folder_name}"
os.makedirs(CONFIG["output_dir"], exist_ok=True)

print(f"Outputs: {CONFIG['output_dir']}")

In [ ]:
class ADE20K_Robust_Dataset(Dataset):
    def __init__(self, split="train", fraction=1.0, processor=None, image_size=512, 
                 use_hf=True, local_root=None):
        self.image_size = image_size
        self.use_hf = use_hf
        self.local_root = local_root
        self.split = split
        self.fraction = fraction
        
        self.image_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        if use_hf:
            self._load_hf_dataset(split, fraction)
        else:
            self._load_local_dataset(split, fraction)
        
        if CONFIG.get("precache", False):
            print("Pre-caching dataset to RAM...")
            self.cached_images = []
            self.cached_labels = []
            for i in tqdm(range(len(self.samples)), desc="Caching"):
                img, label = self._load_item(i)
                self.cached_images.append(img)
                self.cached_labels.append(label)
            print("Caching complete!")
            
    def _load_hf_dataset(self, split, fraction):
        hf_split = "train" if split == "train" else "validation"
        print(f"Loading HF dataset: scene_parse_150 ({hf_split})...")
        self.hf_data = load_dataset("scene_parse_150", split=hf_split)
        if fraction < 1.0:
            n = max(1, int(len(self.hf_data) * fraction))
            self.hf_data = self.hf_data.shuffle(seed=42).select(range(n))
        print(f"Loaded {len(self.hf_data)} HF image-mask pairs")
        
    def _load_local_dataset(self, split, fraction):
        print(f" Loading local dataset from {self.local_root}...")

        if split == "train":
            img_dir = os.path.join(self.local_root, "images", "training")
            ann_dir = os.path.join(self.local_root, "annotations", "training")
        else:
            img_dir = os.path.join(self.local_root, "images", "validation")
            ann_dir = os.path.join(self.local_root, "annotations", "validation")
            
        if not os.path.isdir(img_dir) or not os.path.isdir(ann_dir):
            raise FileNotFoundError(f"Expected dirs not found:\n  {img_dir}\n  {ann_dir}")
            
        # Collect all images
        img_files = glob.glob(os.path.join(img_dir, "*.jpg")) + \
                    glob.glob(os.path.join(img_dir, "*.jpeg")) + \
                    glob.glob(os.path.join(img_dir, "*.png"))
                    
        print(f"Found {len(img_files)} images. Matching with PNG masks...")
        
        self.samples = []
        for img_path in img_files:
            base = os.path.splitext(os.path.basename(img_path))[0]
            ann_path = os.path.join(ann_dir, f"{base}.png")
            
            if os.path.exists(ann_path):
                self.samples.append((img_path, ann_path))
                
        if not self.samples:
            raise FileNotFoundError(
                f" No matching PNG masks found.\n"
                f"Images: {img_dir}\nAnnotations: {ann_dir}"
            )
            
        print(f"Loaded {len(self.samples)} image-mask pairs")
        
        if fraction < 1.0:
            n = max(1, int(len(self.samples) * fraction))
            import random
            random.seed(42)
            self.samples = random.sample(self.samples, n)
            print(f" Using {n}/{len(self.samples)} samples ({fraction*100:.0f}%)")
    
    def __len__(self):
        return len(self.hf_data) if self.use_hf else len(self.samples)
    
    def _load_item(self, idx):
        """Helper method to load a single item without processor"""
        img_path, ann_path = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        seg = Image.open(ann_path)
        return image, seg
    
    def _process_annotation(self, seg_pil):
        """Convert ADE20K annotation to labels (0-149 valid, -100 ignore)"""
        seg_np = np.array(seg_pil, dtype=np.int64)
        
        if seg_np.ndim == 3 and seg_np.shape[2] == 3:
            seg_np = seg_np[:, :, 0].astype(np.int64)
        
        label = seg_np.copy()
        label[label == 0] = -100                       # background → ignore
        valid_mask = (label >= 1) & (label <= 150)
        label[valid_mask] = label[valid_mask] - 1      # 1..150 → 0..149
        label[(label < 0) & (label != -100)] = -100
        label[label > 149] = -100
        return label
    
    def __getitem__(self, idx):
        try:
            # Use cached version if available
            if CONFIG.get("precache", False) and hasattr(self, 'cached_images'):
                image = self.cached_images[idx]
                seg = self.cached_labels[idx]
            else:
                if self.use_hf:
                    item = self.hf_data[idx]
                    image = item["image"].convert("RGB")
                    seg = item["annotation"]
                else:
                    img_path, ann_path = self.samples[idx]
                    image = Image.open(img_path).convert("RGB")
                    seg = Image.open(ann_path)
            
            
            image = image.resize((self.image_size, self.image_size), Image.BILINEAR)
            seg = seg.resize((self.image_size, self.image_size), Image.NEAREST)
            
           
            pixel_values = self.image_transform(image)
            
           
            label = self._process_annotation(seg)
            label = torch.from_numpy(label).long()
            
            return {"pixel_values": pixel_values, "labels": label}
            
        except Exception as e:
            print(f"Error loading sample {idx}: {e}")
            return {
                "pixel_values": torch.zeros(3, self.image_size, self.image_size),
                "labels": torch.full((self.image_size, self.image_size), -100, dtype=torch.long)
            }

In [ ]:
import sys
import os
import torch
import urllib.request


applications_path = '/kaggle/working/efficientvit/applications/efficientvit_seg'
if applications_path not in sys.path:
    sys.path.append(applications_path)

print("Building EfficientViT-B2 architecture and loading pretrained weights...")

weights_dir = "/kaggle/working/assets/checkpoints/efficientvit_seg"
os.makedirs(weights_dir, exist_ok=True)
weight_path = os.path.join(weights_dir, "efficientvit_seg_b2_ade20k.pt")

if not os.path.exists(weight_path):
    print(f"Downloading pretrained weights to {weight_path}...")
    weight_url = "https://huggingface.co/han-cai/efficientvit-seg/resolve/main/efficientvit_seg_b2_ade20k.pt"
    try:
        urllib.request.urlretrieve(weight_url, weight_path)
        print("Download complete.")
    except Exception as e:
        raise FileNotFoundError(f"Manual download required from {weight_url} to {weight_path}")
else:
    print("Using cached pretrained weights.")


from efficientvit.models.efficientvit import efficientvit_seg_b2


model = efficientvit_seg_b2(dataset='ade20k')
print("Model architecture built.")
state_dict = torch.load(weight_path, map_location='cpu')
if 'state_dict' in state_dict:
    state_dict = state_dict['state_dict']
elif 'model' in state_dict:
    state_dict = state_dict['model']

from collections import OrderedDict
new_state_dict = OrderedDict()
for k, v in state_dict.items():
    name = k[7:] if k.startswith('module.') else k
    new_state_dict[name] = v

model.load_state_dict(new_state_dict, strict=True)
print("Pretrained weights loaded successfully.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model = model.to(device)

print("Model is ready for training.")

In [ ]:
import torch
from torch.utils.data import DataLoader

print("\nCreating streaming datasets...")

train_ds = ADE20K_Robust_Dataset(
    split="train",
    fraction=CONFIG["data_split"],
    processor=None,
    image_size=CONFIG["image_size"],
    use_hf=DATASET_SOURCE=="hf",
    local_root=ADE20K_ROOT if DATASET_SOURCE=="local" else None
)

val_ds = ADE20K_Robust_Dataset(
    split="val",
    fraction=CONFIG.get("val_fraction", 1.0),
    processor=None,
    image_size=CONFIG["image_size"],
    use_hf=DATASET_SOURCE=="hf",
    local_root=ADE20K_ROOT if DATASET_SOURCE=="local" else None
)

train_loader = DataLoader(
    train_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],          
    pin_memory=CONFIG["pin_memory"],            
    persistent_workers=CONFIG["persistent_workers"], 
    prefetch_factor=CONFIG["prefetch_factor"],  
    drop_last=True                              
)

val_loader = DataLoader(
    val_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=2,                              
    pin_memory=True,
    persistent_workers=False
)

print(f"Train: {len(train_loader)} batches | Val: {len(val_loader)} batches")

In [ ]:
import time
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from tqdm.notebook import tqdm
import os
import gc
from torch.cuda.amp import GradScaler, autocast



def compute_miou_batch(preds, labels, num_classes=150, ignore_index=-100):
    """Vectorized mIoU + per-class breakdown."""
    preds = preds.flatten()
    labels = labels.flatten()
    valid = labels != ignore_index
    preds, labels = preds[valid], labels[valid]
    if len(labels) == 0:
        return 0.0, {}
    
    intersection = torch.bincount(
        num_classes * labels + preds, minlength=num_classes**2
    ).reshape(num_classes, num_classes).diagonal()
    
    union = (torch.bincount(labels, minlength=num_classes) + 
             torch.bincount(preds, minlength=num_classes) - intersection)
    
    ious = intersection.float() / (union.float() + 1e-10)
    per_class = {i: ious[i].item() if union[i] > 0 else -1 for i in range(num_classes)}
    
    valid_ious = ious[union > 0]
    miou = valid_ious.mean().item() if len(valid_ious) > 0 else 0.0
    return miou, per_class
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print(f"GPU memory cleared. Free: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB used")

optimizer = AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
total_steps = len(train_loader) * CONFIG["num_epochs"]
warmup_steps = int(total_steps * CONFIG["warmup_ratio"])
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

scaler = GradScaler(enabled=CONFIG["use_amp"])
torch.backends.cudnn.benchmark = True

checkpoint_path = f"{CONFIG['output_dir']}/checkpoint.pt"
start_epoch, best_miou = 0, 0.0
history = {"train_loss": [], "val_miou": [], "per_class_miou": []}

print(f"\nStarting training: {CONFIG['num_epochs']} epochs | {int(CONFIG['data_split']*100)}pct data")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")
print(f"AMP enabled: {CONFIG['use_amp']}")
print("="*60)

for epoch in range(start_epoch, CONFIG["num_epochs"]):
    elapsed_hours = (time.time() - CONFIG["session_start"]) / 3600
    if elapsed_hours > CONFIG["max_session_hours"]:
        print(f"Approaching timeout ({elapsed_hours:.1f}h). Saving & exiting.")
        break

    model.train()
    total_loss, batch_count = 0.0, 0
    progress = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1} [Train]")

    epoch_start = time.time()
    for batch_idx, batch in progress:
        pixel_values = batch["pixel_values"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        with autocast(enabled=CONFIG["use_amp"]):
            outputs = model(pixel_values)
            if isinstance(outputs, dict):
                logits = outputs.get('seg', outputs.get('out'))
                if logits is None:
                    logits = next(v for v in outputs.values() if torch.is_tensor(v))
            elif torch.is_tensor(outputs):
                logits = outputs
            else:
                raise TypeError(f"Unexpected model output: {type(outputs)}")

            logits = F.interpolate(logits, size=labels.shape[-2:], mode='bilinear', align_corners=False)
            loss = F.cross_entropy(logits, labels, ignore_index=-100)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["max_grad_norm"])
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()
        batch_count += 1
        progress.set_postfix({"loss": f"{loss.item():.4f}", "avg": f"{total_loss/batch_count:.4f}"})
        

        if batch_idx % 50 == 0:
            del outputs, logits, loss
            torch.cuda.empty_cache()

    avg_loss = total_loss / batch_count
    history["train_loss"].append(avg_loss)
    print(f"\nTraining completed in {(time.time()-epoch_start)/60:.1f} min | Avg Loss: {avg_loss:.4f}")

    model.eval()
    all_preds, all_labels = [], []
    val_start = time.time()
    
    with torch.no_grad(), tqdm(val_loader, desc="Validating", leave=False) as pbar:
        for batch_idx, batch in enumerate(pbar):
            pixel_values = batch["pixel_values"].to(device, non_blocking=True)
            labels = batch["labels"]
            
            with autocast(enabled=CONFIG["use_amp"]):
                outputs = model(pixel_values)
                if isinstance(outputs, dict):
                    logits = outputs.get('seg', outputs.get('out'))
                    if logits is None:
                        logits = next(v for v in outputs.values() if torch.is_tensor(v))
                else:
                    logits = outputs
                logits = F.interpolate(logits, size=labels.shape[-2:], mode='bilinear', align_corners=False)

            preds = logits.argmax(dim=1).cpu().detach()
            all_preds.append(preds)
            all_labels.append(labels)
            
            
            del outputs, logits, pixel_values
            if batch_idx % 20 == 0:
                torch.cuda.empty_cache()
                gc.collect()

    if all_preds:
        all_preds_cat = torch.cat(all_preds).cpu()
        all_labels_cat = torch.cat(all_labels).cpu()
        miou, per_class_ious = compute_miou_batch(all_preds_cat, all_labels_cat, CONFIG["num_classes"])
        history["per_class_miou"].append(per_class_ious)
        del all_preds_cat, all_labels_cat
    else:
        miou = 0.0
    
    history["val_miou"].append(miou)
    print(f"Validation completed in {(time.time()-val_start)/60:.1f} min | mIoU: {miou*100:.2f}%")

    del all_preds, all_labels
    torch.cuda.empty_cache()
    gc.collect()

    if miou > best_miou:
        best_miou = miou
        best_dir = f"{CONFIG['output_dir']}/best_model"
        os.makedirs(best_dir, exist_ok=True)
        torch.save(model.state_dict(), f"{best_dir}/pytorch_model.bin")
        print(f" New best mIoU: {best_miou*100:.2f}%")

    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scaler_state": scaler.state_dict(),
        "best_miou": best_miou,
        "history": history
    }, checkpoint_path)
    print(f"Checkpoint saved to: {checkpoint_path}")

print(f"\n Training Complete! Best mIoU: {best_miou*100:.2f}%")

print("\n Generating paper-ready outputs...")

results_path = f"{CONFIG['output_dir']}/results.json"
with open(results_path, "w") as f:
    json.dump({
        "model": "EfficientViT-B2",
        "data_split": CONFIG["data_split"],
        "best_miou": best_miou,
        "history": history
    }, f, indent=2)
print(f"Results saved to: {results_path}")

viz_dir = f"{CONFIG['output_dir']}/visualizations"
os.makedirs(viz_dir, exist_ok=True)

split_tag = f"{int(CONFIG['data_split']*100)}pct"

plt.figure(figsize=(6, 4))
plt.plot(history["train_loss"], color="#2E86AB", linewidth=2, marker='o', markersize=4)
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.title(f"Training Loss ({split_tag} data)", fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()
loss_curve_path = f"{viz_dir}/loss_curve_{split_tag}.png"
plt.savefig(loss_curve_path, dpi=300)
plt.close()
print(f"Loss curve saved: {loss_curve_path}")

plt.figure(figsize=(6, 4))
miou_pct = [m * 100 for m in history["val_miou"]]
plt.plot(miou_pct, color="#A23B72", linewidth=2, marker='s', markersize=4)
plt.axhline(y=best_miou*100, color='red', linestyle='--', label=f'Best: {best_miou*100:.2f}%')
plt.xlabel("Epoch")
plt.ylabel("mIoU (%)")
plt.title(f"Validation mIoU ({split_tag} data)", fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
miou_curve_path = f"{viz_dir}/miou_curve_{split_tag}.png"
plt.savefig(miou_curve_path, dpi=300)
plt.close()
print(f"mIoU curve saved: {miou_curve_path}")

if history["per_class_miou"]:
    per_class_df = pd.DataFrame(
        history["per_class_miou"][-1].items(), 
        columns=["Class_ID", "IoU"]
    )
    per_class_csv_path = f"{viz_dir}/per_class_iou_{split_tag}.csv"
    per_class_df.to_csv(per_class_csv_path, index=False)
    print(f"Per-class IoU saved: {per_class_csv_path}")

print("Generating sample predictions...")
model.eval()
sample_idxs = np.random.choice(len(val_ds), min(5, len(val_ds)), replace=False)

cmap = plt.cm.get_cmap('tab20', CONFIG["num_classes"])
cmap.set_bad(color='black')

for i, idx in enumerate(sample_idxs):
    sample = val_ds[idx]
    with torch.no_grad():
        px = sample["pixel_values"].unsqueeze(0).to(device)
        with autocast(enabled=CONFIG["use_amp"]):
            outputs = model(px)
            if isinstance(outputs, dict):
                logits = outputs.get('seg', outputs.get('out'))
                if logits is None:
                    logits = next(v for v in outputs.values() if torch.is_tensor(v))
            else:
                logits = outputs
        logits = F.interpolate(logits, size=sample["labels"].shape[-2:], mode='bilinear', align_corners=False)
        pred = logits.argmax(1).squeeze(0).cpu()

    fig, ax = plt.subplots(1, 3, figsize=(14, 4.5))

    img_np = sample["pixel_values"].cpu().permute(1, 2, 0).numpy()
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img_np = img_np * std + mean
    img_np = np.clip(img_np, 0, 1)
    ax[0].imshow(img_np)
    ax[0].set_title("Input Image", fontweight='bold')
    ax[0].axis("off")

    pred_vis = pred.float().numpy()
    pred_vis[pred == -100] = np.nan
    ax[1].imshow(pred_vis, cmap=cmap, vmin=0, vmax=CONFIG["num_classes"]-1)
    ax[1].set_title("Prediction", fontweight='bold')
    ax[1].axis("off")
   
    true_vis = sample["labels"].float().numpy()
    true_vis[true_vis == -100] = np.nan
    ax[2].imshow(true_vis, cmap=cmap, vmin=0, vmax=CONFIG["num_classes"]-1)
    ax[2].set_title("Ground Truth", fontweight='bold')
    ax[2].axis("off")
    
    plt.tight_layout()
    sample_path = f"{viz_dir}/sample_{split_tag}_{i:02d}.png"
    plt.savefig(sample_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"   Saved sample {i+1}/5: {sample_path}")

print(f"\nAll outputs saved to: {CONFIG['output_dir']}")
print(f"Visualizations folder: {viz_dir}")